[<img src="https://gitlab.irit.fr/toc/etu-n7/controle-optimal/-/raw/master/ressources/Logo-toulouse-inp-N7.png" alt="N7" height="80"/>](https://gitlab.irit.fr/toc/etu-n7/controle-optimal)
<img src="https://gitlab.irit.fr/toc/ens-n7/texCoursN7/-/raw/main/logo-insa.png" alt="INSA" height="80" style="margin-left:50px"/>

# Systèmes dynamiques

- Date : 2026-2027
- Durée approximative : 2h

**Objectifs.** À l'issue de ce TP, vous savez :

- mettre un système physique ou biologique sous la forme d'un problème à
  valeur initiale $\dot{x}(t) = f(t, x(t))$ ;
- intégrer numériquement une EDO avec `OrdinaryDiffEqTsit5` et lire un portrait
  de phase ;
- observer numériquement des phénomènes qualitatifs (cycle limite, résonance,
  sensibilité aux conditions initiales, chaos).

**Plan.**

1. Pendule simple
2. Oscillateur de Van der Pol
3. Modèle proie-prédateur (Lotka-Volterra)
4. Phénomène de résonance
5. Chaos : attracteur de Lorenz

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Introduction
</div>

On étudie ici quelques modèles classiques d'équations différentielles
ordinaires, énumérés ci-dessus.

Un problème à valeur initiale (IVP, *initial value problem*) est une équation
différentielle ordinaire (EDO) munie d'une condition initiale :
$$
  (IVP) \left\{\begin{array}{l}
    \dot{x}(t) = f(t, x(t)) \\
    x(t_0) = x_0.
  \end{array}\right.
$$
Pour chaque modèle, on écrit le second membre $f$, on intègre le problème
numériquement avec `OrdinaryDiffEqTsit5`, puis on étudie le portrait de phase et,
quand c'est pertinent, la stabilité des points d'équilibre par linéarisation
(jacobienne de $f$ au point d'équilibre, valeurs propres via `ForwardDiff` et
`LinearAlgebra`).

In [ ]:
# activation du projet situé dans le répertoire de ce notebook
using Pkg
Pkg.activate(@__DIR__)

# chargement des paquets
using ForwardDiff
using LinearAlgebra
using OrdinaryDiffEqTsit5
using Plots

# figures vectorielles (nettes à tout zoom)
default(fmt = :svg)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Pendule simple
</div>

On s'intéresse ici au [pendule simple](https://fr.wikipedia.org/wiki/Pendule_simple).
Les principes de la mécanique classique donnent comme équation régissant l'évolution
de l'angle $\alpha$

$$ ml^2\ddot{\alpha}(t) + mlg\sin(\alpha(t)) + k\dot{\alpha}(t) = 0,$$

où $\ddot{\alpha}(t)$ désigne la dérivée seconde de l'angle $\alpha$ par rapport au
temps $t$.

1. En prenant comme variable d'état $x=(x_1,x_2)=(\alpha, \dot{\alpha})$, écrire la
   fonction $f$ qui permet d'écrire l'équation différentielle sous la forme
   $\dot{x}(t) = f(t,x(t))$.
2. Coder cette fonction ci-dessous et exécuter le code. Le mouvement observé est-il
   une oscillation ou une rotation du pendule ?
3. Remplacer la vitesse angulaire initiale $\dot{\alpha}(0)$ par $2$. Commentaires.

In [ ]:
"""
    Second membre de l'IVP
    x : vecteur d'état
    λ : vecteur de paramètres
    t : variable de temps. Ici le temps n'intervient pas explicitement, le système est autonome.
"""
function pendule(x, λ, t)
    xpoint = similar(x)
    g, l, k, m = λ
    xpoint[1] = 0.0; xpoint[2] = 0.0          # à compléter : second membre du pendule
    return xpoint
end

In [ ]:
# intégration et affichage
g = 9.81
l = 10
k = 0
m = 1
λ = [g, l, k, m] # paramètres constants

theta0 = pi/3
x0 = [theta0, 0] # état initial

t0 = 0
tf = 3*pi*sqrt(l/g)*(1 + theta0^2/16 + theta0^4/3072) # 2*approximation de la période
tspan = (t0, tf) # instants initial et terminal

prob = ODEProblem(pendule, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8) # définition du problème en Julia
sol = solve(prob, Tsit5()) # intégration numérique

plot(sol, label = ["x₁(t)" "x₂(t)"]) # affichage de la solution

**Diagramme de phases.**

1. Exécuter le code ci-dessous et interpréter le graphique : où sont les oscillations,
   les rotations et les points d'équilibre stables et instables ?
2. On considère le cas où $k=0.15$ (on introduit un frottement). Que se passe-t-il ?

In [ ]:
#
g = 9.81
l = 1.5
k = 0.15
m = 1
λ = [g, l, k, m] # paramètres constants

plt = plot(xlabel = "x₁", ylabel = "x₂", legend = false) # initialisation du plot

for theta0 in 0:(2*pi)/10:2*pi
    tf = 3*pi*sqrt(l/g)*(1 + theta0^2/16 + theta0^4/3072) # 2*approximation de la période
    tspan = (0.0, tf)
    x0 = [theta0, 0]
    prob = ODEProblem(pendule, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(plt, sol, idxs=(1,2), color="blue")
end

theta0 = pi-10*eps()
x0 = [theta0, 0]
tf = 50                              # problème pour tf=50 (1/4 de la période !)
tspan = (0.0, tf)
prob = ODEProblem(pendule, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
sol = solve(prob, Tsit5())
plot!(plt, sol, idxs=(1,2), xlims = (-2*pi,4*pi), color="green")

theta0 = pi+10*eps()
x0 = [theta0, 0]
tf = 50
tspan = (0.0, tf)
prob = ODEProblem(pendule, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
sol = solve(prob, Tsit5())
plot!(plt, sol, idxs=(1,2), xlims = (-2*pi,4*pi), color="green")

# cas de rotation (circulation)
for thetapoint0 in 0:1.:4
    tf = 10
    tspan = (0.0, tf)
    x0 = [-pi, thetapoint0]               # thetapoint0 > 0 : theta croît de -pi à ...
    prob = ODEProblem(pendule, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(plt, sol, idxs=(1,2), color="red")
end
for thetapoint0 in -4:1.:0
    tf = 10
    tspan = (0.0, tf)
    x0 = [3*pi, thetapoint0]              # thetapoint0 < 0 : theta décroît de 3pi à ...
    prob = ODEProblem(pendule, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(plt, sol, idxs=(1,2), color="purple")
end
plot!(plt, [-pi, 0, pi, 2*pi, 3*pi], [0, 0, 0, 0, 0], seriestype=:scatter)
plot!(plt, xlims = (-pi,3*pi), size=(650, 350))

**Stabilité.**

Exécuter les cellules ci-dessous, qui calculent les valeurs propres de la matrice
jacobienne du second membre du pendule (sans frottement, $k=0$) aux deux points
d'équilibre $x_e = (0,0)$ (position verticale basse) et $x_e = (\pi,0)$ (position
verticale haute). Commentaires : quelle est la nature de chaque équilibre (centre,
col…), et comment cela se lit-il sur le diagramme de phases ci-dessus ?

In [ ]:
# Jacobienne de la fonction pendule (k=0)
λ = [g, l, 0, m]
dfdx(x) = ForwardDiff.jacobian(x -> pendule(x, λ, 0), x)

# Point d'équilibre (0,0) et jacobienne en ce point
xe = [0.0, 0.0]
A = dfdx(xe)

# Valeurs propres de la jacobienne
eigvals(A)

In [ ]:
# Point d'équilibre (π,0) et jacobienne en ce point
xe = [π, 0.0]
A = dfdx(xe)

# Valeurs propres de la jacobienne
eigvals(A)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Oscillateur de Van der Pol
</div>

L'équation différentielle considérée est l'[équation de Van der Pol](https://fr.wikipedia.org/wiki/Oscillateur_de_Van_der_Pol)

$$(IVP)\left\{\begin{array}{l}
\dot{x}_1(t)=x_2(t)\\
\dot{x}_2(t)=(1-x_1^2(t))\, x_2(t)-x_1(t)\\
x_1(0)=2.009\\
x_2(0)=0
\end{array}\right.
$$

jusqu'au temps $t_f=T=6.66$, où $T$ est la période de la
solution.

Coder le second membre de l'IVP ci-dessous et exécuter le code. Commentaires.

In [ ]:
"""
    Second membre de l'IVP : modèle de Van der Pol
    x : vecteur d'état
    λ : vecteur de paramètres (inutilisé)
    t : variable de temps. Ici le temps n'intervient pas explicitement, le système est autonome.
"""
function vdp(x, λ, t)
    xpoint = similar(x)
    xpoint[1] = 0.0; xpoint[2] = 0.0          # à compléter : second membre de Van der Pol
    return xpoint
end

In [ ]:
# intégration et affichage : trajectoires intérieures, extérieures, cycle limite
t0 = 0
tf = 6.66
tspan = (t0, tf)

#
plot(xlabel = "x₁", ylabel = "x₂", legend = false)

# trajectoires intérieures
for x01 in -2:0.4:0
    x0 = [x01, 0]
    prob = ODEProblem(vdp, x0, tspan, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(sol, idxs=(1,2), color = "blue")
end

# trajectoires extérieures
for x02 in 2.5:0.5:4
    x0 = [0, x02]
    prob = ODEProblem(vdp, x0, tspan, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(sol, idxs=(1,2), color = "green")
end

# cycle limite périodique
x0 = [2.009, 0]
prob = ODEProblem(vdp, x0, tspan, reltol = 1.e-8, abstol = 1.e-8)
sol = solve(prob, Tsit5())
plot!(sol, idxs=(1,2), color = "red", lw = 2.)

#
plot!([0], [0], seriestype=:scatter)        # point d'équilibre
plot!(xlims = (-2.5, 2.5), ylims = (-3, 5), size=(650, 350))

**Stabilité.**

On calcule ci-dessous les valeurs propres de la matrice jacobienne du second membre de
l'IVP au point d'équilibre $x_e = (0, 0)$. Commentaires (pour les étudiants N7, se
rappeler du cours d'[automatique](https://gitlab.irit.fr/toc/etu-n7/controle-optimal/-/raw/master/ressources/notes-autom-2021.pdf)).

In [ ]:
# Jacobienne de la fonction vdp
dvdp(x) = ForwardDiff.jacobian(x -> vdp(x, [], 0), x)

# Point d'équilibre et jacobienne en ce point
xe = [0, 0]
A = dvdp(xe)

# Valeurs propres de la jacobienne
eigvals(A)

**Généralisation : le paramètre $\mu$.**

Plus généralement, l'oscillateur de Van der Pol est régi par
$$\ddot{x}_1 - \mu(1-x_1^2)\dot{x}_1 + x_1 = 0,$$
ce qui redonne le second membre codé plus haut pour $\mu = 1$. Le paramètre
$\mu > 0$ règle la force de l'amortissement non linéaire : plus $\mu$ est
grand, plus le cycle limite devient « raide » (relaxation rapide, allure de
créneaux) plutôt qu'une oscillation quasi sinusoïdale.

1. Coder ci-dessous le second membre paramétré par $\mu$.
2. Essayer $\mu = 0.5$ puis $\mu = 5$, avec la condition initiale $x_0=(1,0)$
   et $t_f=40$ (la période exacte donnée plus haut n'est valable que pour
   $\mu=1$ ; on prend donc ici un temps final générique, assez grand pour
   atteindre le cycle limite). Comparer l'allure du cycle limite aux deux
   valeurs.

On choisit ici $x_0 = (1,0)$, qui n’est pas un point du cycle limite, afin de mieux observer la convergence vers celui-ci.

In [ ]:
"""
    Second membre de l'IVP : Van der Pol paramétré par μ
    x : vecteur d'état
    λ : vecteur de paramètres, λ = [μ]
    t : variable de temps. Ici le temps n'intervient pas explicitement, le système est autonome.
"""
function vdpμ(x, λ, t)
    xpoint = similar(x)
    μ = λ[1]
    xpoint[1] = 0.0; xpoint[2] = 0.0          # à compléter : second membre de Van der Pol paramétré par μ
    return xpoint
end

In [ ]:
# intégration et affichage pour deux valeurs de μ
t0 = 0
tf = 40
tspan = (t0, tf)
x0 = [1, 0]

plt = plot(xlabel = "x₁", ylabel = "x₂", legend = :topright)

for μ in [0.5, 5]
    λ = [μ]
    prob = ODEProblem(vdpμ, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(plt, sol, idxs=(1,2), label = "μ = $μ")
end

plt

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Modèle proie-prédateur (Lotka-Volterra)
</div>

L'équation différentielle considérée est donnée par les [équations de prédation de
Lotka-Volterra](https://fr.wikipedia.org/wiki/Équations_de_prédation_de_Lotka-Volterra)

$$\left\{\begin{array}{l}
\dot{x}_1(t)= \phantom{-}x_1(t) ( \alpha - \beta x_2(t))  \\[0.5em]
\dot{x}_2(t)= -x_2(t) ( \gamma - \delta x_1(t))
\end{array}\right.
$$

où

- $t$ est le temps ;
- $x_1(t)$ est l'effectif des proies en fonction du temps ;
- $x_2(t)$ est l'effectif des prédateurs en fonction du temps.

Les paramètres suivants caractérisent les interactions entre les deux espèces :

- $\alpha$, taux de reproduction intrinsèque des proies (constant, indépendant du
  nombre de prédateurs) ;
- $\beta$, taux de mortalité des proies dû aux prédateurs rencontrés ;
- $\delta$, taux de reproduction des prédateurs en fonction des proies rencontrées et
  mangées ;
- $\gamma$, taux de mortalité intrinsèque des prédateurs (constant, indépendant du
  nombre de proies).

1. Coder le second membre de l'IVP ci-dessous.
2. Le point $(0, 0)$ est clairement un point d'équilibre stable. Il existe un autre
   point d'équilibre, lequel ? Le compléter ci-dessous.
3. Exécuter le code et afficher l'autre point d'équilibre ainsi que quelques
   trajectoires du système dans le plan de phase, qui rentrent dans les limites du
   plot ci-dessous. Qu'observe-t-on sur les trajectoires ?
4. Les valeurs propres de la jacobienne au second point d'équilibre sont imaginaires
   pures $\pm i\sqrt{\alpha\gamma}$. Pour observer concrètement la période, ajouter
   une cellule qui trace $x_1(t)$ et $x_2(t)$ en fonction du temps, en partant d'une
   condition initiale proche du second point d'équilibre. Modifier ensuite $\alpha$
   et $\gamma$ dans la cellule du dessus (par exemple en doublant une valeur) et
   comparer la période observée à $2\pi/\sqrt{\alpha\gamma}$. Cette formule décrit
   la période des petites oscillations autour de l'équilibre ; prendre une
   perturbation suffisamment petite pour rester dans ce régime.

In [ ]:
"""
    Second membre de l'IVP : modèle de Lotka-Volterra
    x : vecteur d'état
    λ : vecteur de paramètres
    t : variable de temps. Ici le temps n'intervient pas explicitement, le système est autonome.
"""
function lv(x, λ, t)
    xpoint = similar(x)
    α, β, γ, δ = λ
    xpoint[1] = 0.0; xpoint[2] = 0.0          # à compléter : second membre de Lotka-Volterra
    return xpoint
end

In [ ]:
# intégration et affichage de quelques trajectoires
α = 2/3
β = 4/3
γ = 1
δ = 1
λ = [α, β, γ, δ]

t0 = 0
tf = 20
tspan = (t0, tf)

plt = plot(xlabel = "x₁", ylabel = "x₂", legend = false)

for x02 ∈ range(0.6, 2.5, length = 10)
    x0 = [1, x02]
    prob = ODEProblem(lv, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
    sol = solve(prob, Tsit5())
    plot!(plt, sol, idxs=(1,2), color=:blue)
end

plt

In [ ]:
# second point d'équilibre
xe = [0.0, 0.0]                              # à compléter : second point d'équilibre
xe

In [ ]:
# affichage du second point d'équilibre et des limites du graphique
plot!(plt, [xe[1]], [xe[2]], seriestype=:scatter)
plot!(plt, xlims = (0, 4), ylims = (0, 2.5), size=(650, 350))

In [ ]:
# Jacobienne de la fonction lv au second point d'équilibre
dfdx(x) = ForwardDiff.jacobian(x -> lv(x, λ, 0), x)
A = dfdx(xe)

# Valeurs propres de la jacobienne : comparer à ±i√(αγ) (question 4)
eigvals(A)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Phénomène de résonance
</div>

Soit $\omega$ et $\omega_0$ deux réels strictement positifs. On considère l'équation
différentielle

$$\ddot{y}(t) + \omega_0^2 y(t) = \cos(\omega t).$$

1. En prenant comme variable d'état $x=(x_1,x_2)=(y, \dot{y})$, écrire la fonction $f$
   qui permet d'écrire l'équation différentielle sous la forme $\dot{x}(t) = f(t,x(t))$.
2. Coder cette fonction ci-dessous et exécuter le code avec $\omega_0 = \omega/2$ : les
   solutions sont-elles bornées ?
3. Remplacer la pulsation propre pour avoir $\omega_0 = \omega$. Commentaires.
4. Prendre $\omega_0$ proche de $\omega$ mais différent, par exemple
   $\omega_0 = 0.95\,\omega$ : observer le phénomène de battement. Comment la
   période des battements dépend-elle de $|\omega - \omega_0|$ ?

Remarque : voir la page Wikipedia sur le [phénomène de résonance](https://fr.wikipedia.org/wiki/Résonance)
pour plus d'informations.

In [ ]:
"""
    Second membre de l'IVP
    x : vecteur d'état
    λ : vecteur de paramètres
    t : variable de temps. Ici le temps intervient explicitement, le système est non autonome.
"""
function resonance(x, λ, t)
    xpoint = similar(x)
    ω, ω₀ = λ
    xpoint[1] = 0.0; xpoint[2] = 0.0          # à compléter : second membre de l'oscillateur forcé
    return xpoint
end

In [ ]:
# paramètres : essayer ω₀ = ω/2 (question 2, bornée), ω₀ = ω (question 3,
# résonance) ou ω₀ = 0.95*ω (question 4, quasi-résonance / battements)
ω  = 1
ω₀ = ω
λ  = [ω, ω₀]

# instants d'intégration
t0 = 0
tf = 140
tspan = (t0, tf)

# condition initiale
x0 = [0, 0]

# définition et résolution du problème
prob = ODEProblem(resonance, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
sol = solve(prob, Tsit5())

# construit et affiche une animation de la trajectoire dans le plan de phase
# (x₁,x₂), avec le temps porté sur le troisième axe
function makegif()

    #
    plt = plot(xlabel = "x₁", ylabel = "x₂", zlabel = "t", legend = false)

    #
    lim = ω == ω₀ ? 50 : (abs(ω - ω₀) < 0.1*ω ? 25 : 5)  # résonance / quasi-résonance / bornée
    plot!(xlims = (-lim, lim), ylims = (-lim, lim), zlims = (t0, tf), size=(850, 580))

    # barre de progression sobre, sous le graphique (piste dessinée une fois pour
    # toutes ; le remplissage, opaque, est redessiné par-dessus à chaque image)
    plot!(plt, inset = bbox(0.06, 0.015, 0.88, 0.01, :bottom, :left), subplot=2,
          bg_inside=:white, framestyle=:none, legend=false)
    plot!(plt[2], Shape([0,1,1,0],[0,0,1,1]), color=RGBA(0,0,0,0.08), linewidth=0, subplot=2)

    # récupération de x1, x2, t depuis sol
    x1 = [sol.u[i][1] for i ∈ 1:length(sol.u)]
    x2 = [sol.u[i][2] for i ∈ 1:length(sol.u)]
    t  = sol.t
    n  = length(sol.t)

    # tracé pas à pas de la trajectoire et fabrication d'un gif
    i = 1
    @gif for j ∈ 2:2:n
        plot!(plt, x1[i:j], x2[i:j], t[i:j], color=:blue)
        i = j
        plot!(plt[2], Shape([0,j/n,j/n,0],[0,0,1,1]), color=:blue, linewidth=0, subplot=2)
    end

end

makegif()

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Chaos : attracteur de Lorenz
</div>

L'équation différentielle considérée est l'[équation de Lorenz](https://fr.wikipedia.org/wiki/Attracteur_de_Lorenz)
donnée par

$$\left\{\begin{array}{l}
\dot{x}_1(t)= \phantom{-}\sigma (x_2(t) - x_1(t))  \\[0.5em]
\dot{x}_2(t)= \rho\, x_1(t) - x_2(t) - x_1(t)\, x_3(t)  \\[0.5em]
\dot{x}_3(t)= x_1(t)\, x_2(t) - \beta x_3(t)
\end{array}\right.
$$

où

- $t$ est le temps ;
- $x_1(t)$ est proportionnel à l'intensité du mouvement de convection ;
- $x_2(t)$ est proportionnel à la différence de température entre les courants
  ascendants et descendants ;
- $x_3(t)$ est proportionnel à la déviation du profil vertical de température par
  rapport à la valeur linéaire de référence ;
- $\sigma$ (nombre de Prandtl), $\rho$ (nombre de Rayleigh) et $\beta$ sont des
  paramètres réels positifs.

On fixe souvent $\sigma = 10$ et $\beta = 8/3$ ; le paramètre $\rho$ pilote alors le
comportement qualitatif du système, avec trois régimes séparés par les valeurs
critiques $\rho = 1$ et $\rho_H = \sigma(\sigma+\beta+3)/(\sigma-\beta-1) \approx 24{,}74$ :

- $\rho < 1$ : l'origine est l'unique équilibre, il est stable ;
- $1 < \rho < \rho_H$ : l'origine devient instable, les deux équilibres non
  triviaux $(\pm\sqrt{\beta(\rho-1)}, \pm\sqrt{\beta(\rho-1)}, \rho-1)$ sont des
  foyers stables (spirales qui convergent) ;
- $\rho > \rho_H$ : ces deux équilibres deviennent instables — c'est le régime
  chaotique, illustré ici avec $\rho = 28$.

1. Compléter le code ci-dessous et l'exécuter avec $\rho = 28$. Observer le
   phénomène de chaos, en modifiant éventuellement la condition initiale.
2. Reprendre avec $\rho = 0.5$, puis $\rho = 10$ : le comportement de la
   trajectoire par rapport aux trois points d'équilibre affichés est-il
   cohérent avec les trois régimes ci-dessus ?

In [ ]:
"""
    Second membre de l'IVP : modèle de Lorenz
    x : vecteur d'état
    λ : vecteur de paramètres
    t : variable de temps. Ici le temps n'intervient pas explicitement, le système est autonome.
"""
function lorenz(x, λ, t)
    xpoint = similar(x)
    σ, ρ, β = λ
    xpoint[1] = 0.0; xpoint[2] = 0.0; xpoint[3] = 0.0   # à compléter : second membre de Lorenz
    return xpoint
end

In [ ]:
# paramètres
σ = 10
ρ = 28
β = 8/3
λ = [σ, ρ, β]

# instants d'intégration
t0 = 0
tf = 100
tspan = (t0, tf)

# condition initiale
x0 = [0, 2, 0]

# définition et résolution du problème
prob = ODEProblem(lorenz, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8)
sol = solve(prob, Tsit5())

# construit et affiche une animation 3D de la trajectoire (x₁,x₂,x₃), avec les
# trois points d'équilibre repérés en rouge
function makegif()

    #
    plt = plot(xlabel = "x₁", ylabel = "x₂", zlabel = "x₃", legend = false)
    plot!(plt, background_color = :royalblue4)
    plot!(plt, axiscolor = :white, tickfontcolor = :white, bordercolor = :white, fg_color_guide = :white)

    # les trois points d'équilibre (les deux non triviaux n'existent que pour ρ>1)
    plot!(plt, [0], [0], [0], seriestype=:scatter, color=:red)
    if ρ > 1
        plot!(plt, [ sqrt(β*(ρ-1))], [ sqrt(β*(ρ-1))], [ρ-1], seriestype=:scatter, color=:red)
        plot!(plt, [-sqrt(β*(ρ-1))], [-sqrt(β*(ρ-1))], [ρ-1], seriestype=:scatter, color=:red)
    end

    # récupération de x1, x2, x3 depuis sol
    x1 = [sol.u[i][1] for i ∈ 1:length(sol.u)]
    x2 = [sol.u[i][2] for i ∈ 1:length(sol.u)]
    x3 = [sol.u[i][3] for i ∈ 1:length(sol.u)]

    #
    plot!(xlims = (-20, 20), ylims = (-25, 30), zlims = (0, 50), size=(850, 580))

    # barre de progression sobre, sous le graphique (piste dessinée une fois pour
    # toutes ; le remplissage, opaque, est redessiné par-dessus à chaque image)
    plot!(plt, inset = bbox(0.06, 0.015, 0.88, 0.01, :bottom, :left), subplot=2,
          bg_inside=:royalblue4, framestyle=:none, legend=false)
    plot!(plt[2], Shape([0,1,1,0],[0,0,1,1]), color=RGBA(1,1,1,0.15), linewidth=0, subplot=2)

    #
    duration = 10
    fps      = 10
    frames   = fps*duration
    step     = length(sol.t) ÷ frames
    n        = length(sol.t)

    if step == 0
        return
    end

    # tracé pas à pas de la trajectoire et fabrication d'un gif
    i = 1
    animation = @animate for j ∈ 2:step:n
        plot!(plt, x1[i:j], x2[i:j], x3[i:j], color=:gold2)
        i = j
        plot!(plt[2], Shape([0,j/n,j/n,0],[0,0,1,1]), color=:gold2, linewidth=0, subplot=2)
    end

    gif(animation, "lorenz.gif", fps=fps)

end

makegif()

**L'attracteur attire (presque) toute condition initiale.**

Le système de Lorenz est dissipatif : la divergence de son second membre vaut
$-\sigma - 1 - \beta$, une constante strictement négative, donc tout volume de
conditions initiales se contracte exponentiellement au cours du temps. On peut
même montrer qu'il existe une région bornée absorbante que toute trajectoire finit
par atteindre, quel que soit son point de départ dans $\mathbb{R}^3$. Dans le
régime chaotique ($\rho=28$), cette contraction fait donc converger *presque*
toutes les trajectoires vers le même ensemble, l'attracteur de Lorenz — à
l'exception d'un ensemble de mesure nulle (la variété stable du point-selle à
l'origine, qui elle converge vers $(0,0,0)$).

Pour l'observer, on relance l'intégration depuis `n_points` conditions initiales
dispersées aléatoirement et on affiche simultanément leurs trajectoires (avec une
traînée courte plutôt que l'historique complet, pour rester lisible) : quel que
soit leur point de départ, elles finissent toutes par se coller sur le même
papillon à deux ailes.

In [ ]:
# --- paramètres ---
n_points = 100   # nombre de trajectoires (conditions initiales)
tf       = 1.5   # durée simulée
tail_len = 0.15  # longueur de la traînée affichée, même unité que tf (0 <= tail_len <= tf ;
                 # 0 = seulement le marqueur de position courante, sans traînée)
ralenti  = 1     # facteur de ralenti de l'animation (ralenti = 2 -> deux fois plus lente)
zoom     = 1.2   # facteur de zoom des axes et de la boîte de départ des conditions initiales
                 # (zoom > 1 : dézoome, on part plus loin et on voit plus large ; zoom < 1 : zoome)

@assert 0 <= tail_len <= tf "tail_len doit être compris entre 0 et tf"
@assert ralenti > 0 "ralenti doit être strictement positif"
@assert zoom > 0 "zoom doit être strictement positif"

# --- paramètres techniques (densité d'échantillonnage, images par seconde de
# référence) : réglés pour un rendu lisse, pas besoin d'y toucher ---
n_saved       = 1000
fps_base      = 15
duration_base = 20

t0 = 0
tspan = (t0, tf)
t_grid = range(t0, tf, length = n_saved)
tail_len_idx = round(Int, tail_len/tf * n_saved)   # traînée en nombre de points sauvegardés

# demi-largeur en x,y et hauteur en z de la boîte de départ et des axes (référence :
# x₁,x₂ ∈ [-25,25], x₃ ∈ [0,50], l'extension habituelle de l'attracteur)
box_xy = 25*zoom
box_z  = 50*zoom

# conditions initiales dispersées aléatoirement, loin de l'attracteur
x0s = [[2*box_xy*rand() - box_xy, 2*box_xy*rand() - box_xy, box_z*rand()] for _ in 1:n_points]

# intégration de chaque trajectoire, échantillonnée sur la même grille de temps fine
# (saveat) : ça lisse les courbes tracées et permet d'empiler les trajectoires dans
# des matrices communes (une colonne par trajectoire)
sols = [solve(ODEProblem(lorenz, x0, tspan, λ, reltol = 1.e-8, abstol = 1.e-8, saveat = t_grid), Tsit5()) for x0 in x0s]
X1 = reduce(hcat, [s[1, :] for s in sols])   # n_saved × n_points
X2 = reduce(hcat, [s[2, :] for s in sols])
X3 = reduce(hcat, [s[3, :] for s in sols])

# construit et affiche une animation 3D, au ralenti : n_points trajectoires
# (traînée courte, ou simple marqueur si tail_len = 0) convergeant vers
# l'attracteur, quelle que soit leur condition initiale
function makegif_many()

    frames = fps_base * duration_base   # nombre d'images, indépendant de ralenti
    step   = n_saved ÷ frames
    if step == 0
        return
    end
    fps = fps_base / ralenti            # ralentit la lecture sans recalculer d'images

    animation = @animate for j in step:step:n_saved
        i = max(1, j - tail_len_idx)

        plt = plot(xlabel = "x₁", ylabel = "x₂", zlabel = "x₃", legend = false, size=(850,580))
        plot!(plt, background_color = :royalblue4)
        plot!(plt, axiscolor = :white, tickfontcolor = :white, bordercolor = :white, fg_color_guide = :white)

        # les trois points d'équilibre (les deux non triviaux n'existent que pour ρ>1)
        plot!(plt, [0], [0], [0], seriestype=:scatter, color=:red)
        if ρ > 1
            plot!(plt, [ sqrt(β*(ρ-1))], [ sqrt(β*(ρ-1))], [ρ-1], seriestype=:scatter, color=:red)
            plot!(plt, [-sqrt(β*(ρ-1))], [-sqrt(β*(ρ-1))], [ρ-1], seriestype=:scatter, color=:red)
        end

        # traînée récente de chaque trajectoire
        if tail_len_idx > 0
            plot!(plt, X1[i:j, :], X2[i:j, :], X3[i:j, :], color=:gold2)
        end
        # position courante de chaque trajectoire (marqueur, toujours affiché)
        plot!(plt, X1[j:j, :], X2[j:j, :], X3[j:j, :], seriestype=:scatter,
              color=:gold2, markersize=2, markerstrokewidth=0)

        plot!(plt, xlims = (-box_xy, box_xy), ylims = (-box_xy, box_xy), zlims = (0, box_z))

        # barre de progression sobre, sous le graphique (plt est reconstruit à
        # chaque image, donc piste et remplissage sont redessinés ensemble)
        plot!(plt, inset = bbox(0.06, 0.015, 0.88, 0.01, :bottom, :left), subplot=2,
              bg_inside=:royalblue4, framestyle=:none, legend=false)
        plot!(plt[2], Shape([0,1,1,0],[0,0,1,1]), color=RGBA(1,1,1,0.15), linewidth=0, subplot=2)
        plot!(plt[2], Shape([0,j/n_saved,j/n_saved,0],[0,0,1,1]), color=:gold2, linewidth=0, subplot=2)
    end

    gif(animation, "lorenz_many.gif", fps=fps)

end

makegif_many()

**Sensibilité aux conditions initiales.**

(Reprendre $\rho = 28$ si une autre valeur a été testée à la question précédente : la
suite suppose le régime chaotique.)

Le système de Lorenz est l'exemple historique de « l'effet papillon » : deux
conditions initiales infiniment proches donnent, après un temps fini, deux
trajectoires complètement différentes — bien que le système soit
déterministe (pas de bruit, pas de hasard).

1. Reprendre l'intégration ci-dessous avec la condition initiale perturbée
   $x_0' = x_0 + \varepsilon\, e_1$, $\varepsilon = 10^{-8}$ (perturbation
   uniquement sur $x_1$).
2. Tracer sur un même graphique $x_1(t)$ pour les deux trajectoires ($x_0$ et
   $x_0'$). Jusqu'à quel instant environ les deux courbes sont-elles
   indiscernables ? Que se passe-t-il ensuite ?
3. Tracer $\log_{10}\|x(t) - x'(t)\|$ en fonction de $t$ : la croissance
   est-elle compatible avec une divergence exponentielle (comportement
   affine par morceaux sur l'échelle log) ?